# Notebook 6: Prescriptive Feature Engineering

## 1. Introduction
This notebook focuses on the **Feature Engineering** phase of the Prescriptive Analytics engine. 

The goal is to create a robust mechanism that can take raw resource inputs (such as the number of workers, working hours, and machines) and transform them into the exact feature format required by the saved `production_model.joblib`. 

### Key Objectives:
1. **Consistency**: Ensure that the features generated for prescriptions are identical in structure to those used during model training.
2. **Dynamic Calculation**: Implement logic to calculate derived features (like worker-machine ratios) on-the-fly for any given scenario.
3. **Pipeline Verification**: Validate that the generated features are correctly processed by the pre-trained pipeline.

In [2]:
import pandas as pd
import numpy as np
import joblib
import os

# Load the saved model pipeline
MODEL_PATH = '../models/production_model.joblib'
model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")

Model loaded successfully.


## 2. Feature Builder Implementation

We need a function that handles the conversion from raw business inputs to a model-ready DataFrame. This function must include the derived features used in Phase 1: `machine_worker_ratio` and `avg_hours_per_worker`.

In [5]:
def build_prescriptive_features(raw_inputs):
    """
    Transforms raw resource inputs into a DataFrame for the production model.
    Ensures all columns required by the pipeline are present.
    """
    # Create base dictionary
    features = raw_inputs.copy()

    # 1. Calculate Dynamic Derived Features
    workers = max(raw_inputs.get('no_of_workers', 1), 1)
    features['machine_worker_ratio'] = raw_inputs.get('machine_count', 0) / workers
    features['avg_hours_per_worker'] = raw_inputs.get('working_hours', 0) / workers

    # 2. Fill missing baseline context columns with defaults if not provided
    # These are required by the model but aren't part of the resource optimization
    defaults = {
        'week_no': 26,
        'week_of_month': 3,
        'financial_month': 'Sep',
        'bottleneck_flag': False,
        'rate': 5266.92 # Median rate from EDA
    }

    for col, value in defaults.items():
        if col not in features:
            features[col] = value

    # Convert to DataFrame
    df = pd.DataFrame([features])

    return df

## 3. Verification & Testing

In this section, we test the feature builder with a sample scenario and pass it through the model to ensure no errors occur.

In [6]:
# Sample raw inputs representing a production scenario
sample_scenario = {
    # --- Optimization Variables (Things we change) ---
    'no_of_workers': 15,
    'working_hours': 623.96,
    'machine_count': 4,

    # --- Fixed Context Variables (Required by model) ---
    'item_name': 'DIYANA - L - BROWN',
    'factory_unit': 'Factory Unit 01',
    'product_family': 'Gents',
    'size_id': 10,
    'grade': 'A',
    'order_category': 'Others',
    'week_no': 26,
    'week_of_month': 3,
    'financial_month': 'Sep',
    'bottleneck_flag': False,
    'rate': 5266.92
}
# 1. Build features
features_df = build_prescriptive_features(sample_scenario)
print("Generated Features DataFrame:")
print(features_df[['no_of_workers', 'working_hours', 'machine_count', 'machine_worker_ratio', 'avg_hours_per_worker']])

# 2. Run prediction to verify pipeline consistency
try:
    prediction = model.predict(features_df)
    print(f"\nPrediction successful! Predicted Actual Qty: {prediction[0]:.2f}")
except Exception as e:
    print(f"\nPrediction failed. Error: {e}")

Generated Features DataFrame:
   no_of_workers  working_hours  machine_count  machine_worker_ratio  \
0             15         623.96              4              0.266667   

   avg_hours_per_worker  
0             41.597333  

Prediction successful! Predicted Actual Qty: 35.22


## 4. Summary
The `build_prescriptive_features` function is now ready to be used by the Optimizer in Notebook 8. It ensures that for every simulation, the model receives the correct derived metrics, maintaining the mathematical integrity of the predictive engine.